In [9]:
import json
import pandas as pd
from collections import defaultdict

INPUT_JSONL = "/Users/talhasoylu/Downloads/doccanno0904.jsonl"
OUTPUT_CSV = "/Users/talhasoylu/Downloads/gold_standard1.csv"

MED_LABELS = {"Med", "Classe"}
DOSAGE_LABELS = {"Dosage"}
FREQ_LABELS = {"Freq"}
ROUTE_LABELS = {"Route"}
DATE_LABELS = {"Date"}
DATE_REL_LABELS = {"Date_relative", "Duree"}
COND_LABELS = {"Contexte"}

REL_LINK = {"Refer_to", "Start", "Stop", "Augmentation", "Duration_presc"}

def join_unique(values):
    out = []
    for v in values:
        v = (v or "").strip()
        if v and v not in out:
            out.append(v)
    return " | ".join(out)

rows = []

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        text = item.get("text", "")
        ents = item.get("entities", [])
        rels = item.get("relations", [])

        ents_by_id = {}
        for e in ents:
            ent_id = int(e["id"])
            sid = int(e["start_offset"])
            eid = int(e["end_offset"])
            ents_by_id[ent_id] = {
                "id": ent_id,
                "label": str(e["label"]).strip(),
                "start": sid,
                "end": eid,
                "text": text[sid:eid]
            }

        # médicaments/classes triés dans l’ordre du texte
        meds = sorted(
            [e for e in ents_by_id.values() if e["label"] in MED_LABELS],
            key=lambda x: x["start"]
        )

        links = defaultdict(list)
        for r in rels:
            rtype = str(r.get("type", "")).strip()
            if rtype not in REL_LINK:
                continue

            try:
                a = int(r.get("from_id"))
                b = int(r.get("to_id"))
            except (TypeError, ValueError):
                continue

            if a in ents_by_id and b in ents_by_id:
                if ents_by_id[a]["label"] in MED_LABELS and ents_by_id[b]["label"] not in MED_LABELS:
                    links[a].append(b)
                elif ents_by_id[b]["label"] in MED_LABELS and ents_by_id[a]["label"] not in MED_LABELS:
                    links[b].append(a)

        for med in meds:
            mid = med["id"]

            # éléments liés triés dans l’ordre du texte
            linked = sorted(
                [ents_by_id[x] for x in links.get(mid, []) if x in ents_by_id],
                key=lambda x: x["start"]
            )

            rows.append({
                "Medicament": med["text"],
                "Dosage": join_unique([e["text"] for e in linked if e["label"] in DOSAGE_LABELS]),
                "Frequence": join_unique([e["text"] for e in linked if e["label"] in FREQ_LABELS]),
                "Voie d'administration": join_unique([e["text"] for e in linked if e["label"] in ROUTE_LABELS]),
                "Date": join_unique([e["text"] for e in linked if e["label"] in DATE_LABELS]),
                "Date_relative": join_unique([e["text"] for e in linked if e["label"] in DATE_REL_LABELS]),
                "Condition": join_unique([e["text"] for e in linked if e["label"] in COND_LABELS]),
            })

df = pd.DataFrame(rows, columns=[
    "Medicament", "Dosage", "Frequence",
    "Voie d'administration", "Date", "Date_relative", "Condition"
])

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print("OK ->", OUTPUT_CSV, "lignes:", len(df))

mask = (
    df["Condition"].fillna("").astype(str).str.strip().ne("")
    | df["Date_relative"].fillna("").astype(str).str.strip().ne("")
)
print(df.loc[mask, ["Medicament", "Date", "Date_relative", "Condition"]].head(20))

OK -> /Users/talhasoylu/Downloads/gold_standard1.csv lignes: 89
         Medicament    Date   Date_relative                Condition
3        furosémide                                     Introduction
4          Ramipril                      J3                 suspendu
5        furosémide                                          diminué
6          tramadol                                             Mise
7       paracétamol                                             Mise
8          Tramadol           Après 5 jours  arrêté | repris | arrêt
10       amiodarone  02/04.                             Introduction
11        Warfarine                J7 | 48h        arrêtée | reprise
14       prednisone          7 jours. | J4.                     Mise
15  insuline rapide                                     Introduction
16       Sertraline   10/01                                  débutée
17         zolpidem                                            Ajout
18       Ibuprofène               1 sem